# 03 - Data Cleaning & Transaction Segmentation
## E-commerce Sales Analysis & Business Intelligence Dashboard
**Dataset:** UCI Online Retail II (`Year 2009-2010` & `Year 2010-2011`)  
**Tools:** Python, Pandas, NumPy  

---

### Purpose & Cleaning Strategy
In `02_complete_dataset_audit.ipynb`, we discovered that the raw dataset ($1,067,371$ records) contains multiple distinct transaction types beyond standard retail purchases:
- Exact duplicate log entries ($34,335$ rows).
- Customer cancellations and credit notes ($19,494$ rows prefixed with `'C'`).
- Internal warehouse inventory adjustments ($3,457$ negative-quantity write-downs and $2,551$ positive-quantity stock corrections with zero prices and no customer IDs).
- Accounting debt write-downs ($5$ records with negative prices titled *'Adjust bad debt'*).
- Customer promotional items ($70$ records with zero price attached to registered customer IDs).

#### Core Engineering Principles:
1. **Preserve Raw Lineage:** The raw Excel workbook and initial exploration notebooks remain strictly untouched. We load both sheets into `df_raw` and create an explicit working copy `df`.
2. **Context-Aware Segmentation:** Rather than blindly dropping records, we partition the dataset into purpose-built analytical datasets. Cancellations, inventory adjustments, and accounting entries are preserved in separate tables for domain-specific analyses.
3. **Surgical Deduplication:** Only exact duplicate rows (identical across all 8 fields) are removed. Legitimate multi-line invoice items sharing identical `(Invoice, StockCode)` are retained.
4. **Feature Engineering:** We enrich the cleaned retail transactions with granular revenue and calendar dimensions (`Year`, `Month`, `Month_Name`, `Quarter`, `Day`, `Day_Name`, `Hour`).
5. **Airtight Mathematical Reconciliation:** Every single raw record is accounted for without accidental omission or double counting.

In [1]:
import os
import numpy as np
import pandas as pd

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("Environment initialized: Pandas and NumPy loaded successfully.")

Environment initialized: Pandas and NumPy loaded successfully.


## 1. Load and Combine the Raw Dataset

We load both sheets (`Year 2009-2010` and `Year 2010-2011`) from `data/raw/online_retail_II.xlsx` and concatenate them into `df_raw`. We then create an isolated working copy named `df`.

In [2]:
# Determine path relative to notebooks directory or project root
file_path = '../data/raw/online_retail_II.xlsx' if os.path.exists('../data/raw/online_retail_II.xlsx') else 'data/raw/online_retail_II.xlsx'
print(f"Loading raw dataset from: {file_path}")

# Load both sheets
df_2009_2010 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
print(f"Loaded 'Year 2009-2010': {df_2009_2010.shape[0]:,} rows x {df_2009_2010.shape[1]} columns")

df_2010_2011 = pd.read_excel(file_path, sheet_name='Year 2010-2011')
print(f"Loaded 'Year 2010-2011': {df_2010_2011.shape[0]:,} rows x {df_2010_2011.shape[1]} columns")

# Combine into df_raw
df_raw = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
print(f"Combined df_raw shape:   {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Create working DataFrame
df = df_raw.copy()
print(f"Working DataFrame 'df' initialized with {len(df):,} records.")

Loading raw dataset from: ../data/raw/online_retail_II.xlsx


Loaded 'Year 2009-2010': 525,461 rows x 8 columns


Loaded 'Year 2010-2011': 541,910 rows x 8 columns
Combined df_raw shape:   1,067,371 rows x 8 columns
Working DataFrame 'df' initialized with 1,067,371 records.


## 2. Investigate and Clean Duplicates

A critical distinction must be made between:
1. **Exact Duplicate Rows:** Rows where all 8 columns (`Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`) are identical. These represent redundant system logs or network resubmissions.
2. **Partial Duplicates on `(Invoice, StockCode)`:** Rows sharing the same invoice and product code, but having differing quantities, prices, or timestamps. In e-commerce retail, customers frequently purchase different variations of a SKU or add additional units in separate transaction lines.

We investigate these two groups and remove **only** exact 8-column duplicate records.

In [3]:
# Check duplicates sharing (Invoice, StockCode)
inv_stock_dups = df.duplicated(subset=['Invoice', 'StockCode'], keep=False)
total_inv_stock_dups = inv_stock_dups.sum()

# Check exact 8-column duplicates
exact_dups_mask = df.duplicated(keep=False)
total_exact_dup_rows = exact_dups_mask.sum()
duplicate_rows_to_remove = df.duplicated().sum()

print("--- Duplicate Investigation ---")
print(f"Total rows sharing (Invoice, StockCode): {total_inv_stock_dups:,}")
print(f"Total rows involved in exact duplicates: {total_exact_dup_rows:,}")
print(f"Rows sharing (Invoice, StockCode) with differing details: {total_inv_stock_dups - total_exact_dup_rows:,}")
print(f"Exact duplicate rows to remove (keeping 1st occurrence):  {duplicate_rows_to_remove:,}")

# Sample showing legitimate repeated line items vs exact duplicates
print("\nSample of legitimate repeated line items on same invoice (differing quantities/prices):")
legit_repeats = df[inv_stock_dups & (~exact_dups_mask)].sort_values(by=['Invoice', 'StockCode']).head(6)
display(legit_repeats[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']])

# Remove exact duplicates only
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nWorking DataFrame shape after removing exact duplicates: {df.shape[0]:,} rows x {df.shape[1]} columns")

--- Duplicate Investigation ---
Total rows sharing (Invoice, StockCode): 88,585
Total rows involved in exact duplicates: 67,242
Rows sharing (Invoice, StockCode) with differing details: 21,343
Exact duplicate rows to remove (keeping 1st occurrence):  34,335

Sample of legitimate repeated line items on same invoice (differing quantities/prices):


,Invoice,StockCode,Description,Quantity,Price,Customer ID
289,489488,22125,UNION JACK HOT WATER BOTTLE,2,5.9500,17238.0000
299,489488,22125,UNION JACK HOT WATER BOTTLE,1,5.9500,17238.0000
373,489517,21790,VINTAGE SNAP CARDS,4,0.8500,16329.0000
393,489517,21790,VINTAGE SNAP CARDS,1,0.8500,16329.0000
372,489517,21791,VINTAGE HEADS AND TAILS CARD GAME,6,1.2500,16329.0000
392,489517,21791,VINTAGE HEADS AND TAILS CARD GAME,1,1.2500,16329.0000



Working DataFrame shape after removing exact duplicates: 1,033,036 rows x 8 columns


## 3. Handle Cancelled Transactions

Invoices beginning with `'C'` represent credit notes and transaction cancellations:
- In our audit, $19,494$ rows in the raw data had a `'C'` prefix ($19,104$ remaining post-deduplication).
- Over $96\%$ of these cancelled transactions have valid `Customer ID` tags, proving they reflect genuine customer returns.
- These records are essential for calculating return rates, net sales volume, and customer churn metrics.
- We separate them into `df_cancellations` to preserve their analytical value without contaminating gross sales volume.

In [4]:
# Convert Invoice to string to ensure safe string matching
df['Invoice_str'] = df['Invoice'].astype(str)

# Filter cancelled invoices (prefix 'C')
mask_cancelled = df['Invoice_str'].str.startswith('C')
df_cancellations = df[mask_cancelled].copy()

print("--- Cancelled Transactions Overview ---")
print(f"Total Cancelled Transaction Rows: {len(df_cancellations):,}")
print(f"Unique Cancelled Invoice IDs:     {df_cancellations['Invoice'].nunique():,}")
print(f"Cancelled rows with Customer ID:  {df_cancellations['Customer ID'].notnull().sum():,} ({df_cancellations['Customer ID'].notnull().sum() / len(df_cancellations) * 100:.2f}%)")

print("\nCancellation Quantity Breakdown:")
canc_neg = (df_cancellations['Quantity'] < 0).sum()
canc_pos = (df_cancellations['Quantity'] > 0).sum()
print(f"- Negative Quantity (standard returns/credit notes): {canc_neg:,}")
print(f"- Positive Quantity (manual cancellation entry):      {canc_pos:,}")

# Inspect the 1 manual positive cancellation
if canc_pos > 0:
    print("\nManual positive quantity cancellation:")
    display(df_cancellations[df_cancellations['Quantity'] > 0])

--- Cancelled Transactions Overview ---
Total Cancelled Transaction Rows: 19,104
Unique Cancelled Invoice IDs:     8,292
Cancelled rows with Customer ID:  18,390 (96.26%)

Cancellation Quantity Breakdown:
- Negative Quantity (standard returns/credit notes): 19,103
- Positive Quantity (manual cancellation entry):      1

Manual positive quantity cancellation:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice_str
75972,C496350,M,Manual,1,2010-02-01 08:24:00,373.5700,NaN,United Kingdom,C496350


## 4. Handle Negative Quantities

From our audit, negative quantities fall into two distinct operational groups:
1. **Customer Returns & Credit Notes:** $19,103$ deduplicated rows with `'C'` invoice prefixes (handled in `df_cancellations`).
2. **Internal Warehouse Inventory Adjustments:** $3,393$ deduplicated rows **without** a `'C'` prefix.
   - All $3,393$ of these records have `Price == 0.0` and missing `Customer ID`.
   - Their descriptions include `damages`, `lost`, `broken`, `short`, `check`, `?`, etc.
   - These are internal warehouse stock write-downs rather than customer-facing commerce.

In [5]:
# Inspect non-cancelled negative quantity rows
mask_non_c_neg_qty = (~mask_cancelled) & (df['Quantity'] < 0)
df_non_c_neg_qty = df[mask_non_c_neg_qty]

print("--- Non-Cancelled Negative Quantities ---")
print(f"Count of non-cancelled negative quantity rows: {len(df_non_c_neg_qty):,}")
print(f"Rows with Price == 0.0:                         {(df_non_c_neg_qty['Price'] == 0).sum():,} ({((df_non_c_neg_qty['Price'] == 0).sum() / len(df_non_c_neg_qty)) * 100:.2f}%)")
print(f"Rows with Customer ID not null:                 {df_non_c_neg_qty['Customer ID'].notnull().sum():,}")

print("\nTop 10 Descriptions for Non-Cancelled Negative Quantities:")
display(df_non_c_neg_qty['Description'].value_counts().head(10))

--- Non-Cancelled Negative Quantities ---
Count of non-cancelled negative quantity rows: 3,393
Rows with Price == 0.0:                         3,393 (100.00%)
Rows with Customer ID not null:                 0

Top 10 Descriptions for Non-Cancelled Negative Quantities:


Description
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64

## 5. Handle Negative Prices

Our audit revealed exactly $5$ records with negative unit prices:
- Invoices starting with `'A'` (`A506401`, `A516228`, `A528059`, `A563186`, `A563187`).
- `StockCode: B` and `Description: Adjust bad debt`.
- These represent accounting adjustments writing down uncollectible corporate debt.
- We isolate these into `df_accounting_adj` so they do not distort retail merchandise metrics.

In [6]:
mask_accounting = df['Price'] < 0
df_accounting_adj = df[mask_accounting].copy()

print("--- Accounting Debt Adjustments ---")
print(f"Total Negative Price Records: {len(df_accounting_adj):,}")
display(df_accounting_adj[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID', 'Country']])

--- Accounting Debt Adjustments ---
Total Negative Price Records: 5


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
177338,A506401,B,Adjust bad debt,1,-53594.3600,NaN,United Kingdom
273025,A516228,B,Adjust bad debt,1,-44031.7900,NaN,United Kingdom
398731,A528059,B,Adjust bad debt,1,-38925.8700,NaN,United Kingdom
794040,A563186,B,Adjust bad debt,1,-11062.0600,NaN,United Kingdom
794041,A563187,B,Adjust bad debt,1,-11062.0600,NaN,United Kingdom


## 6. Handle Zero Prices

Post-deduplication, there are $6,014$ records with unit price equal to $£0.00$. Examining these records shows two completely different business contexts:
1. **Internal Inventory Adjustments ($5,944$ rows):** Missing `Customer ID`. Includes both negative stock write-downs ($3,393$ rows) and positive stock corrections ($2,551$ rows, e.g. `found`, `check`, `adjustment`, missing descriptions).
2. **Promotional Customer Giveaways ($70$ rows):** Linked to valid registered `Customer ID`s and standard retail invoices. These represent complimentary items, free samples, or promotional bonuses gifted to customers.

We preserve promotional customer orders in regular sales (with a dedicated `Is_Promotional` flag) and route internal adjustments to `df_inventory_adj`.

In [7]:
mask_zero_price = (df['Price'] == 0) & (~mask_accounting) & (~mask_cancelled)

# Separate promotional vs internal
mask_promotional = mask_zero_price & df['Customer ID'].notnull()
mask_inventory_adj = mask_zero_price & df['Customer ID'].isnull()

df_promotional = df[mask_promotional].copy()
df_inventory_adj = df[mask_inventory_adj].copy()

print("--- Zero Price Classification ---")
print(f"Total Zero-Price Records (Non-Cancelled):           {mask_zero_price.sum():,}")
print(f"- Customer Promotional Items (Valid Customer ID):   {len(df_promotional):,}")
print(f"- Internal Inventory Adjustments (No Customer ID):  {len(df_inventory_adj):,}")

print("\nSample Customer Promotional Records:")
display(df_promotional.head(5)[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']])

print("\nInventory Adjustments Breakdown by Quantity Direction:")
print(f"- Negative Quantity (Stock write-downs/damages/losses): {(df_inventory_adj['Quantity'] < 0).sum():,}")
print(f"- Positive Quantity (Found stock/corrections/samples):  {(df_inventory_adj['Quantity'] > 0).sum():,}")

--- Zero Price Classification ---
Total Zero-Price Records (Non-Cancelled):           6,014
- Customer Promotional Items (Valid Customer ID):   70
- Internal Inventory Adjustments (No Customer ID):  5,944

Sample Customer Promotional Records:


,Invoice,StockCode,Description,Quantity,Price,Customer ID
4607,489825,22076,6 RIBBONS EMPIRE,12,0.0000,16126.0000
6702,489998,48185,DOOR MAT FAIRY CAKE,2,0.0000,15658.0000
15906,490727,M,Manual,1,0.0000,17231.0000
18531,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,0.0000,14108.0000
18532,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,0.0000,14108.0000



Inventory Adjustments Breakdown by Quantity Direction:
- Negative Quantity (Stock write-downs/damages/losses): 3,393
- Positive Quantity (Found stock/corrections/samples):  2,551


## 7. Handle Missing Values & Isolate Regular Sales

With non-regular categories separated, we isolate clean regular retail transactions:
- **Missing `Customer ID`:** Retained in regular sales. Approximately $22.67\%$ of retail transactions lack customer registration (anonymous guest checkouts). Removing them would severely understate total revenue and transaction volume. They are flagged for exclusion during customer-level RFM modeling, but included in sales/revenue reporting.
- **Missing `Description`:** All $4,382$ missing product descriptions observed in the raw audit belonged to warehouse adjustments and were cleanly routed to `df_inventory_adj`. As a result, **our regular retail sales dataset has 0 missing descriptions!**

In [8]:
# Regular sales filter: exclude accounting adjustments, cancellations, and internal inventory adjustments
mask_regular = (~mask_accounting) & (~mask_cancelled) & (~mask_inventory_adj)
df_regular = df[mask_regular].copy().reset_index(drop=True)

print("--- Clean Regular Sales Dataset ---")
print(f"Total Regular Sales Records: {len(df_regular):,}")

# Missing value audit on regular sales
print("\nMissing Values in Clean Regular Sales:")
missing_reg = pd.DataFrame({
    'Missing Count': df_regular.isnull().sum(),
    'Percentage (%)': (df_regular.isnull().sum() / len(df_regular)) * 100
})
display(missing_reg)

print(f"Quantity Range: [{df_regular['Quantity'].min():,}, {df_regular['Quantity'].max():,}]")
print(f"Price Range:    [£{df_regular['Price'].min():.2f}, £{df_regular['Price'].max():.2f}]")

--- Clean Regular Sales Dataset ---
Total Regular Sales Records: 1,007,983

Missing Values in Clean Regular Sales:


,Missing Count,Percentage (%)
Invoice,0,0.0000
StockCode,0,0.0000
Description,0,0.0000
Quantity,0,0.0000
InvoiceDate,0,0.0000
Price,0,0.0000
Customer ID,228488,22.6678
Country,0,0.0000
Invoice_str,0,0.0000


Quantity Range: [1, 80,995]
Price Range:    [£0.00, £25111.09]


## 8. Create Business Features

To prepare the dataset for exploratory data analysis, business intelligence, SQL queries, and Power BI dashboards, we engineer granular commercial and temporal dimensions:
- `Revenue`: Monetary line-item value calculated as `Quantity * Price`.
- `Is_Promotional`: Boolean flag marking the $70$ free customer promotional orders (`Price == 0.0`).
- Date components extracted from `InvoiceDate`:
  - `Year`
  - `Month`
  - `Month_Name`
  - `Quarter`
  - `Day`
  - `Day_Name`
  - `Hour`

We also apply these business features to `df_cancellations` to support downstream return analytics.

In [9]:
# Feature engineering for regular sales
df_regular['Revenue'] = df_regular['Quantity'] * df_regular['Price']
df_regular['Is_Promotional'] = df_regular['Price'] == 0.0

# Temporal features
df_regular['Year'] = df_regular['InvoiceDate'].dt.year
df_regular['Month'] = df_regular['InvoiceDate'].dt.month
df_regular['Month_Name'] = df_regular['InvoiceDate'].dt.strftime('%B')
df_regular['Quarter'] = df_regular['InvoiceDate'].dt.quarter
df_regular['Day'] = df_regular['InvoiceDate'].dt.day
df_regular['Day_Name'] = df_regular['InvoiceDate'].dt.strftime('%A')
df_regular['Hour'] = df_regular['InvoiceDate'].dt.hour

# Also engineer features for cancellations
df_cancellations['Revenue'] = df_cancellations['Quantity'] * df_cancellations['Price']
df_cancellations['Year'] = df_cancellations['InvoiceDate'].dt.year
df_cancellations['Month'] = df_cancellations['InvoiceDate'].dt.month
df_cancellations['Month_Name'] = df_cancellations['InvoiceDate'].dt.strftime('%B')
df_cancellations['Quarter'] = df_cancellations['InvoiceDate'].dt.quarter
df_cancellations['Day'] = df_cancellations['InvoiceDate'].dt.day
df_cancellations['Day_Name'] = df_cancellations['InvoiceDate'].dt.strftime('%A')
df_cancellations['Hour'] = df_cancellations['InvoiceDate'].dt.hour

# Drop temporary Invoice_str column if present
if 'Invoice_str' in df_regular.columns:
    df_regular.drop(columns=['Invoice_str'], inplace=True)
if 'Invoice_str' in df_cancellations.columns:
    df_cancellations.drop(columns=['Invoice_str'], inplace=True)
if 'Invoice_str' in df_inventory_adj.columns:
    df_inventory_adj.drop(columns=['Invoice_str'], inplace=True)
if 'Invoice_str' in df_accounting_adj.columns:
    df_accounting_adj.drop(columns=['Invoice_str'], inplace=True)

print("Feature engineering completed successfully.")
print("\nSample of Enriched Regular Sales Data:")
display(df_regular.head(5))

Feature engineering completed successfully.

Sample of Enriched Regular Sales Data:


C:\Users\sabha\AppData\Local\Temp\ipykernel_26964\1860368216.py:26: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_regular.drop(columns=['Invoice_str'], inplace=True)
C:\Users\sabha\AppData\Local\Temp\ipykernel_26964\1860368216.py:28: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_cancellations.drop(columns=['Invoice_str'], inplace=True)
C:\Users\sabha\AppData\Local\Temp\ipykernel_26964\1860368216.py:30: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_inventory_adj.drop(colu

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Is_Promotional,Year,Month,Month_Name,Quarter,Day,Day_Name,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.9500,13085.0000,United Kingdom,83.4000,False,2009,12,December,4,1,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085.0000,United Kingdom,81.0000,False,2009,12,December,4,1,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085.0000,United Kingdom,81.0000,False,2009,12,December,4,1,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1000,13085.0000,United Kingdom,100.8000,False,2009,12,December,4,1,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.2500,13085.0000,United Kingdom,30.0000,False,2009,12,December,4,1,Tuesday,7


## 9. Validation & Reconciliation Report

We perform a complete mathematical reconciliation across all segments to guarantee that:
- Every record from `df_raw` is accounted for.
- There are no overlapping records between categories.
- Zero rows are unintentionally lost or double-counted.

In [10]:
# Reconciliation audit
raw_count = len(df_raw)
dup_count = duplicate_rows_to_remove
dedup_count = len(df)

canc_count = len(df_cancellations)
inv_count = len(df_inventory_adj)
acc_count = len(df_accounting_adj)
reg_count = len(df_regular)
prom_count = int(df_regular['Is_Promotional'].sum())

segmented_sum = canc_count + inv_count + acc_count + reg_count
reconciled_with_raw = segmented_sum + dup_count

validation_data = [
    {'Metric': 'Original Raw Row Count', 'Count': f"{raw_count:,}", 'Notes': 'Combined Year 2009-2010 and Year 2010-2011'},
    {'Metric': 'Exact Duplicate Rows Removed', 'Count': f"{dup_count:,}", 'Notes': 'Exact 8-column duplicate entries'},
    {'Metric': 'Deduplicated Working Dataset', 'Count': f"{dedup_count:,}", 'Notes': 'Raw rows minus exact duplicates'},
    {'Metric': 'Cancelled Transactions Separated', 'Count': f"{canc_count:,}", 'Notes': 'Invoices prefixed with C (credit notes / returns)'},
    {'Metric': 'Inventory Adjustments Separated', 'Count': f"{inv_count:,}", 'Notes': 'Internal warehouse corrections (no Customer ID, £0)'},
    {'Metric': 'Accounting Adjustments Separated', 'Count': f"{acc_count:,}", 'Notes': 'Negative price entries (Adjust bad debt)'},
    {'Metric': 'Promotional Records Retained', 'Count': f"{prom_count:,}", 'Notes': 'Customer free gifts (£0 price with Customer ID)'},
    {'Metric': 'Final Regular Sales Row Count', 'Count': f"{reg_count:,}", 'Notes': 'Clean commercial sales dataset'},
    {'Metric': 'Remaining Missing Customer IDs', 'Count': f"{df_regular['Customer ID'].isnull().sum():,}", 'Notes': f"{df_regular['Customer ID'].isnull().sum() / len(df_regular) * 100:.2f}% (guest checkouts)"},
    {'Metric': 'Remaining Missing Descriptions', 'Count': f"{df_regular['Description'].isnull().sum():,}", 'Notes': '0.00% (all product descriptions present)'},
    {'Metric': 'Total Regular Sales Revenue', 'Count': f"£{df_regular['Revenue'].sum():,.2f}", 'Notes': 'Gross revenue from regular sales'},
    {'Metric': 'Total Cancellations Value', 'Count': f"-£{abs(df_cancellations['Revenue'].sum()):,.2f}", 'Notes': 'Refunded monetary value'},
    {'Metric': 'Regular Sales Date Range', 'Count': f"{df_regular['InvoiceDate'].min().strftime('%Y-%m-%d')} to {df_regular['InvoiceDate'].max().strftime('%Y-%m-%d')}", 'Notes': '738 days spanning 2 full years'}
]

df_val_report = pd.DataFrame(validation_data)

print("=" * 90)
print("COMPREHENSIVE DATA CLEANING & RECONCILIATION REPORT")
print("=" * 90)
display(df_val_report)

# Explicit mathematical assertion
assert segmented_sum == dedup_count, f"Segmentation mismatch: {segmented_sum} != {dedup_count}"
assert reconciled_with_raw == raw_count, f"Raw mismatch: {reconciled_with_raw} != {raw_count}"

print(f"\n[OK] MATHEMATICAL RECONCILIATION PASSED:")
print(f"  {reg_count:,} (Regular) + {canc_count:,} (Cancelled) + {inv_count:,} (Inventory) + {acc_count:,} (Accounting) = {segmented_sum:,} Deduplicated Rows.")
print(f"  {segmented_sum:,} Deduplicated Rows + {dup_count:,} Exact Duplicates = {raw_count:,} Raw Rows (100.00% Accounted For).")

COMPREHENSIVE DATA CLEANING & RECONCILIATION REPORT


,Metric,Count,Notes
0,Original Raw Row Count,"1,067,371",Combined Year 2009-2010 and Year 2010-2011
1,Exact Duplicate Rows Removed,"34,335",Exact 8-column duplicate entries
2,Deduplicated Working Dataset,"1,033,036",Raw rows minus exact duplicates
3,Cancelled Transactions Separated,"19,104",Invoices prefixed with C (credit notes / returns)
4,Inventory Adjustments Separated,"5,944",Internal warehouse corrections (no Customer ID...
5,Accounting Adjustments Separated,5,Negative price entries (Adjust bad debt)
6,Promotional Records Retained,70,Customer free gifts (£0 price with Customer ID)
7,Final Regular Sales Row Count,"1,007,983",Clean commercial sales dataset
8,Remaining Missing Customer IDs,"228,488",22.67% (guest checkouts)
9,Remaining Missing Descriptions,0,0.00% (all product descriptions present)



[OK] MATHEMATICAL RECONCILIATION PASSED:
  1,007,983 (Regular) + 19,104 (Cancelled) + 5,944 (Inventory) + 5 (Accounting) = 1,033,036 Deduplicated Rows.
  1,033,036 Deduplicated Rows + 34,335 Exact Duplicates = 1,067,371 Raw Rows (100.00% Accounted For).


## 10. Save Processed Datasets

We export the segmented datasets to `data/processed/` using descriptive, standardized file names:
1. `clean_regular_sales.csv`: Core commercial dataset ($1,007,983$ rows) for revenue, product performance, cohort analysis, and Power BI dashboards.
2. `returns_and_cancellations.csv`: Credit notes and cancellations ($19,104$ rows) for return-rate analysis.
3. `inventory_adjustments.csv`: Internal warehouse corrections ($5,944$ rows).
4. `accounting_adjustments.csv`: Corporate bad debt adjustments ($5$ rows).
5. `promotional_orders.csv`: Customer promotional items ($70$ rows).

In [11]:
# Define output directory
output_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
os.makedirs(output_dir, exist_ok=True)

# File paths
path_regular = os.path.join(output_dir, 'clean_regular_sales.csv')
path_cancellations = os.path.join(output_dir, 'returns_and_cancellations.csv')
path_inventory = os.path.join(output_dir, 'inventory_adjustments.csv')
path_accounting = os.path.join(output_dir, 'accounting_adjustments.csv')
path_promotional = os.path.join(output_dir, 'promotional_orders.csv')

print("Saving cleaned datasets to data/processed/...")

df_regular.to_csv(path_regular, index=False)
print(f"[OK] Saved clean regular sales:         {path_regular} ({len(df_regular):,} rows, {os.path.getsize(path_regular) / (1024*1024):.2f} MB)")

df_cancellations.to_csv(path_cancellations, index=False)
print(f"[OK] Saved returns & cancellations:     {path_cancellations} ({len(df_cancellations):,} rows, {os.path.getsize(path_cancellations) / (1024*1024):.2f} MB)")

df_inventory_adj.to_csv(path_inventory, index=False)
print(f"[OK] Saved inventory adjustments:       {path_inventory} ({len(df_inventory_adj):,} rows, {os.path.getsize(path_inventory) / (1024*1024):.2f} MB)")

df_accounting_adj.to_csv(path_accounting, index=False)
print(f"[OK] Saved accounting adjustments:      {path_accounting} ({len(df_accounting_adj):,} rows, {os.path.getsize(path_accounting) / 1024:.2f} KB)")

# Save promotional orders extract
df_regular[df_regular['Is_Promotional']].to_csv(path_promotional, index=False)
print(f"[OK] Saved promotional orders extract:  {path_promotional} ({int(df_regular['Is_Promotional'].sum()):,} rows, {os.path.getsize(path_promotional) / 1024:.2f} KB)")

print("\nAll cleaned datasets saved successfully!")

Saving cleaned datasets to data/processed/...


[OK] Saved clean regular sales:         ../data/processed\clean_regular_sales.csv (1,007,983 rows, 128.57 MB)
[OK] Saved returns & cancellations:     ../data/processed\returns_and_cancellations.csv (19,104 rows, 2.37 MB)
[OK] Saved inventory adjustments:       ../data/processed\inventory_adjustments.csv (5,944 rows, 0.36 MB)
[OK] Saved accounting adjustments:      ../data/processed\accounting_adjustments.csv (5 rows, 0.44 KB)
[OK] Saved promotional orders extract:  ../data/processed\promotional_orders.csv (70 rows, 8.76 KB)

All cleaned datasets saved successfully!


### Final Data Cleaning Summary

| Segment | Final Row Count | Business Purpose & Key Characteristics |
|---|---:|---|
| **Clean Regular Sales** | **1,007,983** | Core commercial retail transactions. Spans 2009-12-01 to 2011-12-09. Total gross revenue: **£20,476,260.45**. 0 missing descriptions; 22.67% anonymous guest checkouts. Includes 70 promotional giveaway items flagged with `Is_Promotional`. |
| **Returns & Cancellations** | **19,104** | Customer returns and credit notes (invoices starting with 'C'). Total refund value: **-£1,462,050.61**. Over 96.26% have valid Customer IDs. Used for return rate analysis and customer satisfaction modeling. |
| **Inventory Adjustments** | **5,944** | Internal warehouse stock adjustments (all have £0 price and missing Customer ID). Contains 3,393 negative write-downs (damages, losses, broken) and 2,551 positive stock reconciliations (found stock, audits). |
| **Accounting Adjustments** | **5** | Corporate financial write-downs for uncollectible bad debt (invoices starting with 'A', negative unit price). Quarantined from retail performance. |
| **Exact Duplicates Removed** | **34,335** | Exact 8-column duplicate records removed from the original 1,067,371 rows. |

---
**Next Step:** Ready for Exploratory Data Analysis (EDA), Cohort Analysis, RFM Customer Segmentation, and Power BI semantic modeling.